# ConsultaAI — Etapa 1.1: Preprocessamento do Dataset C-ORAL-ESQ

Este notebook realiza a segmentação e o resampling de todos os áudios do dataset C-ORAL-ESQ para mono, 16kHz, além de limpar e normalizar as transcrições textuais para gerar os Gabaritos A (coloquial limpo) e B (normalizado).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import soundfile as sf
import matplotlib.pyplot as plt

# Adiciona a pasta src ao sys.path para importar os módulos locais
sys.path.append(str(Path("..").resolve()))

from src.preprocessing.parser import EAFParser
from src.preprocessing.text import clean_text_gabarito_a, normalize_text_gabarito_b
from src.preprocessing.audio import resample_and_mono, slice_audio, save_wav

## 1. Parsing dos Alinhamentos ELAN (.eaf)

Vamos testar a classe `EAFParser` em um arquivo de amostra (`besqau01.eaf`) para ver como os dados são extraídos.

In [ ]:
eaf_sample = Path("../data/input/C-ORAL-ESQ/c-oral-esq_audio_alignment/besqau01/besqau01.eaf")
segments = EAFParser.parse(eaf_sample)
print(f"Total de segmentos no arquivo: {len(segments)}")
print("\nPrimeiros 5 segmentos extraídos:")
for s in segments[:5]:
    print(s)

## 2. Resampling e Slicing de Áudio

Convertemos o áudio original (Estéreo, 22.05kHz) para Mono, 16kHz e fatiamos em pedaços baseados nos segmentos extraídos do EAF.

In [ ]:
wav_sample = Path("../data/input/C-ORAL-ESQ/c-oral-esq_audio_alignment/besqau01/besqau01.wav")
audio_data, sr = resample_and_mono(wav_sample, target_sr=16000)
print("Propriedades do áudio original resamparado:")
print(f"Formato: {audio_data.shape}, Taxa de Amostragem: {sr} Hz, Duração total: {len(audio_data)/sr:.2f} segundos")

# Vamos fatiar o primeiro segmento como teste
first_seg = segments[0]
sliced = slice_audio(audio_data, sr, first_seg["start_ms"], first_seg["end_ms"])
print(f"\nSegmento fatiado de teste: {len(sliced)} samples, Duração: {len(sliced)/sr:.2f}s")

# Plot do sinal de áudio fatiado
plt.figure(figsize=(10, 3))
plt.plot(sliced, color="#2b5c8f")
plt.title(f"Segmento fatiado ({first_seg['speaker']}): '{first_seg['raw_text']}'")
plt.xlabel("Amostras (samples)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

## 3. Limpeza de Transcrições (Gabaritos A e B)

Testamos as regras de limpeza de marcas CHAT e normalização de palavras coloquiais de acordo com a resposta do usuário.

In [ ]:
test_lines = [
    "e como é que tão as questão das vozes //",
    "<hum hum> //",
    "mas nũ me prejudica em nada / que eu nũ dou [/1] faço nada errado que me / fala / pa &fan &van [/3] pa mim / nem nada / né //"
]

for idx, line in enumerate(test_lines):
    print(f"Original {idx+1}: {line}")
    print(f"  Gabarito A: {clean_text_gabarito_a(line)}")
    print(f"  Gabarito B: {normalize_text_gabarito_b(line)}")
    print("-" * 80)

## 4. Execução do Pipeline Completo

Processamos todos os 43 arquivos do dataset C-ORAL-ESQ, salvando os segmentos em `data/processed/audio/` e a tabela de metadados em `data/processed/metadata.csv`.

In [ ]:
raw_txt_dir = Path("../data/input/C-ORAL-ESQ/c-oral-esq_txt")
audio_align_dir = Path("../data/input/C-ORAL-ESQ/c-oral-esq_audio_alignment")
processed_audio_dir = Path("../data/output/audio")
processed_audio_dir.mkdir(parents=True, exist_ok=True)

txt_files = sorted([f for f in os.listdir(raw_txt_dir) if f.endswith(".txt")])
all_processed_segments = []

for txt_file in tqdm(txt_files, desc="Processando arquivos"):
    file_id = txt_file.split(".")[0]
    eaf_path = audio_align_dir / file_id / f"{file_id}.eaf"
    wav_path = audio_align_dir / file_id / f"{file_id}.wav"
    
    if not eaf_path.exists() or not wav_path.exists():
        print(f"Aviso: Arquivos para {file_id} incompletos. EAF ou WAV ausente.")
        continue
        
    # 1. Parse do EAF
    try:
        segments = EAFParser.parse(eaf_path)
    except Exception as e:
        print(f"Erro ao parsear EAF {file_id}: {e}")
        continue
        
    # 2. Carrega e resampla áudio completo
    try:
        audio_data, sr = resample_and_mono(wav_path, target_sr=16000)
    except Exception as e:
        print(f"Erro ao processar áudio {file_id}: {e}")
        continue
        
    # 3. Fatiamento e Processamento de Texto para cada segmento
    for idx, seg in enumerate(segments):
        speaker = seg["speaker"]
        start_ms = seg["start_ms"]
        end_ms = seg["end_ms"]
        raw_text = seg["raw_text"]
        
        # Limpa e normaliza os textos
        gabarito_a = clean_text_gabarito_a(raw_text)
        gabarito_b = normalize_text_gabarito_b(raw_text)
        
        # Fatia o áudio
        sliced_audio = slice_audio(audio_data, sr, start_ms, end_ms)
        
        # Nome do arquivo segmentado
        segment_name = f"{file_id}_{idx:04d}_{speaker}.wav"
        segment_path = processed_audio_dir / segment_name
        
        # Salva o arquivo de áudio fatiado
        try:
            save_wav(segment_path, sliced_audio, sr)
        except Exception as e:
            print(f"Erro ao salvar WAV do segmento {segment_name}: {e}")
            continue
            
        all_processed_segments.append({
            "segment_id": f"{file_id}_{idx:04d}",
            "file_id": file_id,
            "speaker": speaker,
            "start_ms": start_ms,
            "end_ms": end_ms,
            "audio_path": f"data/output/audio/{segment_name}",
            "text_raw": raw_text,
            "text_gabarito_a": gabarito_a,
            "text_gabarito_b": gabarito_b
        })
        
# Salva os metadados em CSV
df = pd.DataFrame(all_processed_segments)
metadata_csv_path = Path("../data/output/metadata.csv")
metadata_csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(metadata_csv_path, index=False, encoding="utf-8")
print(f"\nPipeline concluído! Total de segmentos processados e salvos: {len(df)}")

## 5. Estatísticas do Dataset Processado

Apresentação das estatísticas de duração total do dataset, distribuição por locutor e visualização das primeiras linhas.

In [ ]:
metadata_csv_path = Path("../data/output/metadata.csv")
df = pd.read_csv(metadata_csv_path)

# Calcula duração de cada segmento
df["duration_s"] = (df["end_ms"] - df["start_ms"]) / 1000.0
total_duration_s = df["duration_s"].sum()

print(f"Duração total do dataset fatiado: {total_duration_s:.2f} segundos ({total_duration_s / 3600.0:.2f} horas)")
print(f"Média de duração por segmento: {df['duration_s'].mean():.2f} segundos")
print(f"Total de segmentos: {len(df)}")

print("\nSegmentos por Locutor:")
print(df["speaker"].value_counts())

print("\nExemplos de Metadados processados:")
df[["segment_id", "speaker", "duration_s", "text_gabarito_a", "text_gabarito_b"]].head(10)